In [ ]:
# Task B reviewer audit — Cell 1: paths and input check

from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass  # For use in local Jupyter; adjust PROJECT_ROOT below.

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

TASKB_DIR = PROJECT_ROOT / "journal_experiments/taskB_holdout"
RESULTS_DIR = TASKB_DIR / "results"
PREDICTIONS_DIR = TASKB_DIR / "predictions"
SPLITS_DIR = TASKB_DIR / "splits"

INPUTS = {
    "gold_links": RESULTS_DIR
        / "taskB_authoritative_gold_links_with_split.csv",
    "mention_inputs": RESULTS_DIR
        / "taskB_authoritative_mention_inputs_with_split.csv",
    "link_predictions": RESULTS_DIR
        / "taskB_test_compositional_predictions_locked.csv",
    "link_change_audit": RESULTS_DIR
        / "taskB_test_compositional_change_audit.csv",
    "concept_split": SPLITS_DIR
        / "taskB_definition_concept_holdout_70_15_15.csv",
    "concept_evidence": RESULTS_DIR
        / "taskB_definition_concept_evidence_with_collapse_K100_complete.csv",
    "development_raw_generations": PREDICTIONS_DIR
        / "taskB_definition_development_ISO_generations_all_conditions.csv",
    "test_raw_generations": PREDICTIONS_DIR
        / "taskB_definition_test_ISO_generations_locked.csv",
    "development_canonical_generations": PREDICTIONS_DIR
        / "taskB_definition_development_ISO_generations_prompt_deduplicated.csv",
    "test_canonical_generations": PREDICTIONS_DIR
        / "taskB_definition_test_ISO_generations_prompt_deduplicated.csv",
    "test_definition_metrics": RESULTS_DIR
        / "taskB_definition_test_ISO_detailed_metrics.csv",
}

for name, path in INPUTS.items():
    print(f"{name:36} {'FOUND' if path.is_file() else 'MISSING'}  {path}")

missing = [name for name, path in INPUTS.items() if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "Missing saved Task B artifacts: " + ", ".join(missing)
    )

In [ ]:
# Task B reviewer audit — Cell 2: linking partition composition

import pandas as pd
from IPython.display import display

gold = pd.read_csv(INPUTS["gold_links"])
mentions = pd.read_csv(INPUTS["mention_inputs"])

gold_key = [
    "split",
    "domain",
    "document_key",
    "mention_normalized",
    "concept_normalized_canonical",
]
mention_key = [
    "split",
    "domain",
    "document_key",
    "mention_normalized",
]

for label, frame, required in [
    ("gold links", gold, gold_key),
    ("mention inputs", mentions, mention_key),
]:
    missing_columns = set(required) - set(frame.columns)
    if missing_columns:
        raise ValueError(
            f"{label} is missing columns: {sorted(missing_columns)}"
        )

# Count unique evaluation units, even if a saved table contains
# repeated annotation rows.
gold_unique = gold.drop_duplicates(gold_key)
mentions_unique = mentions.drop_duplicates(mention_key)

document_and_mention_counts = (
    mentions_unique
    .groupby(["split", "domain"])
    .agg(
        documents=("document_key", "nunique"),
        mention_inputs=("mention_normalized", "size"),
    )
    .reset_index()
)

gold_link_counts = (
    gold_unique
    .groupby(["split", "domain"])
    .size()
    .rename("gold_links")
    .reset_index()
)

labels_per_input = (
    gold_unique
    .groupby(mention_key)
    .size()
    .rename("gold_labels")
    .reset_index()
)

multi_label_counts = (
    labels_per_input
    .assign(multi_label=lambda frame: frame["gold_labels"] > 1)
    .groupby(["split", "domain"])["multi_label"]
    .sum()
    .rename("multi_label_inputs")
    .reset_index()
)

partition_composition = (
    document_and_mention_counts
    .merge(
        gold_link_counts,
        on=["split", "domain"],
        validate="one_to_one",
    )
    .merge(
        multi_label_counts,
        on=["split", "domain"],
        validate="one_to_one",
    )
    .sort_values(["split", "domain"])
    .reset_index(drop=True)
)

assert partition_composition.loc[
    partition_composition["split"].eq("test"), "documents"
].sum() == 6

assert partition_composition.loc[
    partition_composition["split"].eq("test"), "mention_inputs"
].sum() == 122

display(partition_composition)

In [ ]:
# Task B reviewer audit — Cell 3 (corrected):
# definition split and evidence composition

concepts = pd.read_csv(INPUTS["concept_split"])
evidence = pd.read_csv(INPUTS["concept_evidence"])

required_concept_columns = {
    "split",
    "concept_key",
    "domain_stratum",
    "support_bin",
    "semantic_link_count",
    "document_count",
}
required_evidence_columns = {
    "split",
    "concept_key",
    "domain",
    "num_article",
    "doi_normalized",
    "evidence_source",
}

for label, frame, required in [
    ("concept split", concepts, required_concept_columns),
    ("concept evidence", evidence, required_evidence_columns),
]:
    missing_columns = required - set(frame.columns)
    if missing_columns:
        raise ValueError(
            f"{label} is missing columns: {sorted(missing_columns)}"
        )

assert not concepts.duplicated(["split", "concept_key"]).any()

concept_counts = concepts.groupby("split").size().to_dict()
assert concept_counts == {
    "train": 220,
    "development": 47,
    "test": 48,
}, concept_counts

domain_distribution = pd.crosstab(
    concepts["domain_stratum"],
    concepts["split"],
)

support_distribution = pd.crosstab(
    concepts["support_bin"],
    concepts["split"],
)

# Construct an audit-only source-document identifier from the
# document fields carried by the saved evidence table.
evidence["source_document"] = (
    evidence["domain"].astype(str)
    + "|"
    + evidence["num_article"].astype(str)
    + "|"
    + evidence["doi_normalized"].fillna("").astype(str)
)

evidence_by_concept = (
    evidence
    .groupby(["split", "concept_key"])
    .agg(
        evidence_rows=("concept_key", "size"),
        evidence_documents=("source_document", "nunique"),
        literal_evidence_rows=(
            "evidence_source",
            lambda values: (values == "literal_occurrence").sum(),
        ),
    )
    .reset_index()
)

definition_audit = concepts.merge(
    evidence_by_concept,
    on=["split", "concept_key"],
    how="left",
    validate="one_to_one",
)

if definition_audit["evidence_rows"].isna().any():
    missing = definition_audit.loc[
        definition_audit["evidence_rows"].isna(),
        ["split", "concept_key"],
    ]
    raise ValueError(
        "Concepts without evidence rows:\n"
        + missing.to_string(index=False)
    )

evidence_distribution = (
    definition_audit
    .groupby("split")
    .agg(
        concepts=("concept_key", "size"),
        mean_gold_links=("semantic_link_count", "mean"),
        median_gold_links=("semantic_link_count", "median"),
        mean_gold_documents=("document_count", "mean"),
        median_gold_documents=("document_count", "median"),
        mean_evidence_rows=("evidence_rows", "mean"),
        median_evidence_rows=("evidence_rows", "median"),
        maximum_evidence_rows=("evidence_rows", "max"),
        mean_evidence_documents=("evidence_documents", "mean"),
        median_evidence_documents=("evidence_documents", "median"),
        maximum_evidence_documents=("evidence_documents", "max"),
    )
    .round(2)
)

print("Concepts by subdomain membership:")
display(domain_distribution)

print("Concepts by gold-link support:")
display(support_distribution)

print("Gold support and available evidence:")
display(evidence_distribution)

In [ ]:
# Task B reviewer audit — Cell 4: linking routes

import ast
import re
import unicodedata
import numpy as np

predictions = pd.read_csv(INPUTS["link_predictions"])

join_columns = [
    "domain",
    "document_key",
    "mention_normalized",
]

prediction_columns = {
    "exact": "predicted_exact_reconstructed",
    "morphological": "predicted_morph_lookup_reconstructed",
    "morphological_open": "predicted_morph_open_reconstructed",
    "decomposition": "predicted_compositional_reconstructed",
}

required = set(join_columns) | set(prediction_columns.values())
missing_columns = required - set(predictions.columns)
if missing_columns:
    raise ValueError(
        f"Locked predictions are missing: {sorted(missing_columns)}"
    )

def normalize_basic(value):
    """Match the normalization used in the original linking evaluation."""
    if pd.isna(value):
        return ""

    value = unicodedata.normalize("NFKC", str(value)).casefold()
    value = (
        value.replace("α", "alpha")
        .replace("β", "beta")
        .replace("γ", "gamma")
    )
    value = re.sub(r"[‐‑‒–—−]", "-", value)
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()

def parse_concept_set(value):
    """Match the original notebook's saved-set parser."""
    if pd.isna(value):
        return set()

    value = str(value).strip()
    if not value:
        return set()

    if value.startswith(("[", "(", "{")):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, (list, tuple, set)):
                return {
                    normalize_basic(item)
                    for item in parsed
                    if normalize_basic(item)
                }
        except (ValueError, SyntaxError):
            pass

    return {
        normalize_basic(item)
        for item in value.split("|")
        if normalize_basic(item)
    }

test_gold_sets = (
    gold_unique.loc[gold_unique["split"].eq("test")]
    .groupby(join_columns)["concept_normalized_canonical"]
    .apply(
        lambda values: {
            normalize_basic(value)
            for value in values
            if normalize_basic(value)
        }
    )
    .rename("gold_set")
    .reset_index()
)

if predictions.duplicated(join_columns).any():
    raise ValueError("Locked predictions have duplicate mention keys.")
# Rebuild gold sets from the authoritative gold-link table.
# Exclude any gold_set column already present in the prediction file.
predictions_for_audit = predictions.drop(
    columns=["gold_set"],
    errors="ignore",
)

linking_audit = predictions_for_audit.merge(
    test_gold_sets,
    on=join_columns,
    how="left",
    validate="one_to_one",
)

assert len(linking_audit) == 122
assert linking_audit["gold_set"].notna().all()

for name, column in prediction_columns.items():
    linking_audit[f"{name}_set"] = (
        linking_audit[column].map(parse_concept_set)
    )

linking_audit["route"] = np.select(
    [
        linking_audit["exact_set"].map(bool),
        linking_audit["morphological_set"].map(bool),
    ],
    [
        "exact_training_lookup",
        "morphological_training_lookup",
    ],
    default="open_normalized_label",
)

route_counts = linking_audit["route"].value_counts().to_dict()
assert route_counts == {
    "exact_training_lookup": 67,
    "morphological_training_lookup": 4,
    "open_normalized_label": 51,
}, route_counts

def linking_scores(frame, prediction_column):
    tp = sum(
        len(gold_set & predicted_set)
        for gold_set, predicted_set
        in zip(frame["gold_set"], frame[prediction_column])
    )
    fp = sum(
        len(predicted_set - gold_set)
        for gold_set, predicted_set
        in zip(frame["gold_set"], frame[prediction_column])
    )
    fn = sum(
        len(gold_set - predicted_set)
        for gold_set, predicted_set
        in zip(frame["gold_set"], frame[prediction_column])
    )

    return {
        "mention_inputs": len(frame),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "micro_F1": (
            2 * tp / (2 * tp + fp + fn)
            if 2 * tp + fp + fn else np.nan
        ),
        "exact_set_matches": sum(
            gold_set == predicted_set
            for gold_set, predicted_set
            in zip(frame["gold_set"], frame[prediction_column])
        ),
    }

route_rows = []

for route, frame in linking_audit.groupby("route"):
    for system, column in [
        ("morphological_open", "morphological_open_set"),
        ("decomposition", "decomposition_set"),
    ]:
        route_rows.append({
            "route": route,
            "system": system,
            **linking_scores(frame, column),
        })

route_results = (
    pd.DataFrame(route_rows)
    .sort_values(["route", "system"])
    .reset_index(drop=True)
)

display(route_results)

In [ ]:
# Task B reviewer audit — Cell 5: gold-concept familiarity

training_concepts = {
    normalize_basic(value)
    for value in gold_unique.loc[
        gold_unique["split"].eq("train"),
        "concept_normalized_canonical",
    ]
    if normalize_basic(value)
}

linking_audit["seen_gold_links"] = linking_audit["gold_set"].map(
    lambda concepts: len(concepts & training_concepts)
)

linking_audit["unseen_gold_links"] = linking_audit["gold_set"].map(
    lambda concepts: len(concepts - training_concepts)
)

linking_audit["gold_familiarity"] = np.select(
    [
        linking_audit["unseen_gold_links"].eq(0),
        linking_audit["seen_gold_links"].eq(0),
    ],
    [
        "all_gold_concepts_seen",
        "all_gold_concepts_unseen",
    ],
    default="mixed_seen_and_unseen",
)

# First show how familiarity and prediction route intersect.
familiarity_counts = (
    linking_audit
    .groupby(["route", "gold_familiarity"])
    .agg(
        mention_inputs=("gold_set", "size"),
        seen_gold_links=("seen_gold_links", "sum"),
        unseen_gold_links=("unseen_gold_links", "sum"),
    )
    .reset_index()
    .sort_values(["route", "gold_familiarity"])
)

display(familiarity_counts)

# Then evaluate each system on these mention-level groups.
familiarity_rows = []

for (route, familiarity), frame in linking_audit.groupby(
    ["route", "gold_familiarity"]
):
    for system, column in [
        ("morphological_open", "morphological_open_set"),
        ("decomposition", "decomposition_set"),
    ]:
        familiarity_rows.append({
            "route": route,
            "gold_familiarity": familiarity,
            "system": system,
            **linking_scores(frame, column),
        })

familiarity_results = (
    pd.DataFrame(familiarity_rows)
    .sort_values(["route", "gold_familiarity", "system"])
    .reset_index(drop=True)
)

display(familiarity_results)

print(
    "Total seen gold links:",
    linking_audit["seen_gold_links"].sum(),
)
print(
    "Total unseen gold links:",
    linking_audit["unseen_gold_links"].sum(),
)

In [ ]:
# Task B reviewer audit — Cell 6: changed linking inputs

linking_audit["prediction_changed"] = [
    baseline != decomposition
    for baseline, decomposition in zip(
        linking_audit["morphological_open_set"],
        linking_audit["decomposition_set"],
    )
]

linking_audit["baseline_exact"] = [
    reference == prediction
    for reference, prediction in zip(
        linking_audit["gold_set"],
        linking_audit["morphological_open_set"],
    )
]

linking_audit["decomposition_exact"] = [
    reference == prediction
    for reference, prediction in zip(
        linking_audit["gold_set"],
        linking_audit["decomposition_set"],
    )
]

linking_audit["change_outcome"] = np.select(
    [
        (
            linking_audit["decomposition_exact"]
            & ~linking_audit["baseline_exact"]
        ),
        (
            linking_audit["baseline_exact"]
            & ~linking_audit["decomposition_exact"]
        ),
    ],
    [
        "improved_to_exact_match",
        "regressed_from_exact_match",
    ],
    default="neither_exact_or_both_exact",
)

changed_inputs = linking_audit.loc[
    linking_audit["prediction_changed"]
].copy()

assert len(changed_inputs) == 13, (
    f"Expected 13 changed inputs; found {len(changed_inputs)}"
)

def show_set(values):
    return " | ".join(sorted(values))

for source, target in [
    ("gold_set", "reference_concepts"),
    ("morphological_open_set", "baseline_prediction"),
    ("decomposition_set", "decomposition_prediction"),
]:
    changed_inputs[target] = changed_inputs[source].map(show_set)

case_columns = [
    "domain",
    "document_key",
    "mention",
    "route",
    "gold_familiarity",
    "reference_concepts",
    "baseline_prediction",
    "decomposition_prediction",
    "baseline_exact",
    "decomposition_exact",
    "change_outcome",
]

# Some saved prediction tables may omit the original display-form mention.
case_columns = [
    column for column in case_columns
    if column in changed_inputs.columns
]

changed_cases_table = (
    changed_inputs[case_columns]
    .sort_values(["change_outcome", "domain", "document_key"])
    .reset_index(drop=True)
)

print("Exact-set outcome counts:")
display(changed_cases_table["change_outcome"].value_counts())

print("All 13 changed inputs:")
with pd.option_context("display.max_colwidth", 120, "display.max_rows", 20):
    display(changed_cases_table)

In [ ]:
# Task B reviewer audit — Cell 7: repeated exact prompts

repeat_audit_rows = []

for split, raw_name, canonical_name, expected_rows in [
    (
        "development",
        "development_raw_generations",
        "development_canonical_generations",
        235,
    ),
    (
        "test",
        "test_raw_generations",
        "test_canonical_generations",
        144,
    ),
]:
    raw = pd.read_csv(INPUTS[raw_name])
    canonical = pd.read_csv(INPUTS[canonical_name])

    required = {
        "generation_id",
        "randomized_call_order",
        "prompt_hash",
        "generated_definition",
    }

    for label, frame in [("raw", raw), ("canonical", canonical)]:
        missing_columns = required - set(frame.columns)
        if missing_columns:
            raise ValueError(
                f"{split} {label} is missing "
                f"{sorted(missing_columns)}"
            )

    assert len(raw) == len(canonical) == expected_rows
    assert raw["generation_id"].is_unique
    assert canonical["generation_id"].is_unique

    raw = raw.copy()
    raw["trimmed_output"] = (
        raw["generated_definition"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    prompt_groups = (
        raw.groupby("prompt_hash")
        .agg(
            calls=("generation_id", "size"),
            distinct_outputs=("trimmed_output", "nunique"),
        )
        .reset_index()
    )

    shared_groups = prompt_groups.loc[
        prompt_groups["calls"] > 1
    ]

    inconsistent_groups = shared_groups.loc[
        shared_groups["distinct_outputs"] > 1
    ]

    # Reconstruct the original notebook's earliest-call rule.
    earliest = (
        raw.sort_values("randomized_call_order")
        .drop_duplicates("prompt_hash", keep="first")
        [["prompt_hash", "trimmed_output"]]
        .rename(columns={"trimmed_output": "earliest_output"})
    )

    raw_with_earliest = raw.merge(
        earliest,
        on="prompt_hash",
        validate="many_to_one",
    )

    changed_rows = int(
        (
            raw_with_earliest["trimmed_output"]
            != raw_with_earliest["earliest_output"]
        ).sum()
    )

    # Verify the reconstructed count against the saved canonical file.
    comparison = raw[
        ["generation_id", "prompt_hash", "trimmed_output"]
    ].merge(
        canonical[
            ["generation_id", "prompt_hash", "generated_definition"]
        ],
        on=["generation_id", "prompt_hash"],
        validate="one_to_one",
    )

    saved_changed_rows = int(
        (
            comparison["trimmed_output"]
            != comparison["generated_definition"]
                .fillna("")
                .astype(str)
                .str.strip()
        ).sum()
    )

    assert changed_rows == saved_changed_rows, (
        split, changed_rows, saved_changed_rows
    )

    repeat_audit_rows.append({
        "split": split,
        "raw_calls": len(raw),
        "unique_prompts": len(prompt_groups),
        "shared_prompt_groups": len(shared_groups),
        "calls_in_shared_groups": int(shared_groups["calls"].sum()),
        "inconsistent_shared_prompt_groups": len(
            inconsistent_groups
        ),
        "calls_in_inconsistent_groups": int(
            inconsistent_groups["calls"].sum()
        ),
        "outputs_changed_by_canonicalization": changed_rows,
    })

repeat_audit = pd.DataFrame(repeat_audit_rows)
display(repeat_audit)

In [ ]:
# Task B reviewer audit — Cell 8: canonicalization effect on corpus BLEU

%pip -q install sacrebleu

from sacrebleu import corpus_bleu
import sacrebleu

test_metrics = pd.read_csv(INPUTS["test_definition_metrics"])
raw_test = pd.read_csv(INPUTS["test_raw_generations"])
canonical_test = pd.read_csv(
    INPUTS["test_canonical_generations"]
)

required_metric_columns = {
    "concept_key",
    "condition",
    "reference_definition",
    "generated_definition",
}

missing_columns = required_metric_columns - set(test_metrics.columns)
if missing_columns:
    raise ValueError(
        f"Test metrics are missing: {sorted(missing_columns)}"
    )

references = (
    test_metrics[["concept_key", "reference_definition"]]
    .drop_duplicates()
)

assert len(references) == 48
assert references["concept_key"].is_unique

bleu_rows = []

for version, generations in [
    ("raw", raw_test),
    ("canonicalized", canonical_test),
]:
    required_generation_columns = {
        "concept_key",
        "condition",
        "generated_definition",
    }
    missing_columns = (
        required_generation_columns - set(generations.columns)
    )
    if missing_columns:
        raise ValueError(
            f"{version} generations are missing: "
            f"{sorted(missing_columns)}"
        )

    assert len(generations) == 144
    assert not generations.duplicated(
        ["condition", "concept_key"]
    ).any()

    scored = generations.merge(
        references,
        on="concept_key",
        how="left",
        validate="many_to_one",
    )

    assert len(scored) == 144
    assert scored["reference_definition"].notna().all()

    for condition, frame in scored.groupby("condition"):
        frame = frame.sort_values("concept_key")

        score = corpus_bleu(
            frame["generated_definition"].fillna("").tolist(),
            [
                frame["reference_definition"]
                .fillna("")
                .tolist()
            ],
        ).score

        bleu_rows.append({
            "version": version,
            "condition": condition,
            "concepts": len(frame),
            "corpus_BLEU_0_to_100": score,
        })

bleu_comparison = (
    pd.DataFrame(bleu_rows)
    .pivot(
        index="condition",
        columns="version",
        values="corpus_BLEU_0_to_100",
    )
)

bleu_comparison["canonicalized_minus_raw"] = (
    bleu_comparison["canonicalized"]
    - bleu_comparison["raw"]
)

print("SacreBLEU version:", sacrebleu.__version__)
display(bleu_comparison.round(4))

In [ ]:
# Task B reviewer audit — Cell 9: combined-score impact

%pip -q install bert-score transformers

import torch
from bert_score import BERTScorer

model_name = (
    "microsoft/"
    "BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
)

device = "cuda" if torch.cuda.is_available() else "cpu"

scorer = BERTScorer(
    model_type=model_name,
    num_layers=12,
    batch_size=16,
    nthreads=4,
    idf=False,
    device=device,
    lang=None,
    rescale_with_baseline=False,
    use_fast_tokenizer=True,
)
scorer._tokenizer.model_max_length = 512

raw_scored = raw_test.merge(
    references,
    on="concept_key",
    how="left",
    validate="many_to_one",
)

raw_scored = raw_scored.sort_values(
    ["condition", "concept_key"]
).reset_index(drop=True)

assert len(raw_scored) == 144
assert raw_scored["reference_definition"].notna().all()

with torch.inference_mode():
    _, _, raw_f1 = scorer.score(
        cands=raw_scored["generated_definition"]
            .fillna("")
            .astype(str)
            .tolist(),
        refs=raw_scored["reference_definition"]
            .fillna("")
            .astype(str)
            .tolist(),
        batch_size=16,
    )

raw_scored["biomedbert_f1"] = raw_f1.cpu().numpy()

# The saved detailed metrics contain the scores used for the
# canonicalized outputs in the submitted analysis.
saved_canonical = (
    test_metrics
    .groupby("condition")["biomedbert_f1"]
    .mean()
    .rename("canonical_mean_biomedbert_f1")
)

raw_means = (
    raw_scored
    .groupby("condition")["biomedbert_f1"]
    .mean()
    .rename("raw_mean_biomedbert_f1")
)

comparison = bleu_comparison.join(raw_means).join(
    saved_canonical
)

comparison["raw_combined"] = (
    comparison["raw"] / 100
    + comparison["raw_mean_biomedbert_f1"]
) / 2

comparison["canonical_combined"] = (
    comparison["canonicalized"] / 100
    + comparison["canonical_mean_biomedbert_f1"]
) / 2

comparison["canonical_minus_raw_combined"] = (
    comparison["canonical_combined"]
    - comparison["raw_combined"]
)

display(
    comparison[
        [
            "raw_mean_biomedbert_f1",
            "canonical_mean_biomedbert_f1",
            "raw_combined",
            "canonical_combined",
            "canonical_minus_raw_combined",
        ]
    ].round(6)
)

In [ ]:
# Task B reviewer audit — Cell 10: linking document sensitivity

sensitivity_rows = []

for omitted_document in sorted(
    linking_audit["document_key"].unique()
):
    retained = linking_audit.loc[
        linking_audit["document_key"] != omitted_document
    ]

    baseline = linking_scores(
        retained,
        "morphological_open_set",
    )
    decomposition = linking_scores(
        retained,
        "decomposition_set",
    )

    sensitivity_rows.append({
        "omitted_document": omitted_document,
        "retained_documents": retained["document_key"].nunique(),
        "retained_mentions": len(retained),
        "baseline_micro_F1": baseline["micro_F1"],
        "decomposition_micro_F1": decomposition["micro_F1"],
        "decomposition_minus_baseline": (
            decomposition["micro_F1"]
            - baseline["micro_F1"]
        ),
    })

document_sensitivity = (
    pd.DataFrame(sensitivity_rows)
    .sort_values("omitted_document")
    .reset_index(drop=True)
)

assert len(document_sensitivity) == 6
assert (document_sensitivity["retained_documents"] == 5).all()

display(document_sensitivity.round(4))

print(
    "Difference range:",
    round(
        document_sensitivity[
            "decomposition_minus_baseline"
        ].min(),
        4,
    ),
    "to",
    round(
        document_sensitivity[
            "decomposition_minus_baseline"
        ].max(),
        4,
    ),
)

In [ ]:
# Task B reviewer audit — Cell 11: definition concept sensitivity

from sacrebleu import corpus_bleu

definition_rows = test_metrics.loc[
    test_metrics["condition"].isin(
        ["COLLAPSE_K50", "UNRANKED_CONTEXTS"]
    )
].copy()

required = {
    "condition",
    "concept_key",
    "generated_definition",
    "reference_definition",
    "biomedbert_f1",
}
missing_columns = required - set(definition_rows.columns)
if missing_columns:
    raise ValueError(
        f"Test metrics are missing: {sorted(missing_columns)}"
    )

assert len(definition_rows) == 96
assert definition_rows["concept_key"].nunique() == 48
assert not definition_rows.duplicated(
    ["condition", "concept_key"]
).any()

def combined_for_condition(frame):
    frame = frame.sort_values("concept_key")

    bleu = corpus_bleu(
        frame["generated_definition"].fillna("").tolist(),
        [frame["reference_definition"].fillna("").tolist()],
    ).score

    mean_biomedbert = frame["biomedbert_f1"].mean()

    return (bleu / 100 + mean_biomedbert) / 2

sensitivity_rows = []

for omitted_concept in sorted(
    definition_rows["concept_key"].unique()
):
    retained = definition_rows.loc[
        definition_rows["concept_key"] != omitted_concept
    ]

    collapse_score = combined_for_condition(
        retained.loc[
            retained["condition"].eq("COLLAPSE_K50")
        ]
    )
    unranked_score = combined_for_condition(
        retained.loc[
            retained["condition"].eq("UNRANKED_CONTEXTS")
        ]
    )

    sensitivity_rows.append({
        "omitted_concept": omitted_concept,
        "retained_concepts": 47,
        "collapse_combined": collapse_score,
        "unranked_combined": unranked_score,
        "collapse_minus_unranked": (
            collapse_score - unranked_score
        ),
    })

definition_sensitivity = pd.DataFrame(sensitivity_rows)

print(
    "Leave-one-concept-out difference range:",
    round(
        definition_sensitivity[
            "collapse_minus_unranked"
        ].min(),
        6,
    ),
    "to",
    round(
        definition_sensitivity[
            "collapse_minus_unranked"
        ].max(),
        6,
    ),
)

print(
    "Number of omissions yielding a positive difference:",
    int(
        (
            definition_sensitivity[
                "collapse_minus_unranked"
            ] > 0
        ).sum()
    ),
    "of",
    len(definition_sensitivity),
)

display(
    definition_sensitivity
    .sort_values("collapse_minus_unranked")
    .iloc[[0, 1, -2, -1]]
    .round(6)
)

In [ ]:
# Task B reviewer audit — Cell 12: export review materials

import json
from datetime import datetime, timezone

AUDIT_DIR = TASKB_DIR / "reviewer_audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

tables = {
    "linking_partition_composition.csv": partition_composition,
    "definition_domain_distribution.csv":
        domain_distribution.reset_index(),
    "definition_support_distribution.csv":
        support_distribution.reset_index(),
    "definition_evidence_distribution.csv":
        evidence_distribution.reset_index(),
    "linking_route_results.csv": route_results,
    "linking_training_familiarity_counts.csv":
        familiarity_counts,
    "linking_training_familiarity_results.csv":
        familiarity_results,
    "linking_changed_cases.csv": changed_cases_table,
    "generation_repeat_audit.csv": repeat_audit,
    "canonicalization_bleu_comparison.csv":
        bleu_comparison.reset_index(),
    "canonicalization_combined_comparison.csv":
        comparison.reset_index(),
    "linking_leave_one_document_out.csv":
        document_sensitivity,
    "definition_leave_one_concept_out.csv":
        definition_sensitivity,
}

for filename, table in tables.items():
    output_path = AUDIT_DIR / filename
    table.to_csv(output_path, index=False)
    print(filename, len(table), "rows")

methods_record = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "purpose": "Post hoc Task B reviewer audit",
    "original_notebook": "Information_DefinitionGeneration.ipynb",
    "sacrebleu_version_in_audit_runtime":
        sacrebleu.__version__,
    "input_paths": {
        name: str(path)
        for name, path in INPUTS.items()
    },
    "interpretation_limits": [
        "All analyses use the original locked splits and saved outputs.",
        "Leave-one-out checks do not retrain or retune models.",
        "Open canonical-label agreement does not independently verify concept identity.",
        "Automatic definition metrics do not establish biomedical factual accuracy.",
        "No expert definition validation was performed in this audit.",
    ],
}

record_path = AUDIT_DIR / "audit_methods.json"
record_path.write_text(
    json.dumps(methods_record, indent=2),
    encoding="utf-8",
)

print("Methods record:", record_path)
print("Audit directory:", AUDIT_DIR)

In [ ]:
# Training-seen versus unseen link-level performance

import pandas as pd

training_rows = gold[
    gold["split"].astype(str).str.lower().isin(["train", "training"])
]
seen_labels = set(training_concepts)

assert seen_labels, f"No training rows found. Split values: {gold['split'].unique()}"

rows = []
for system, prediction_column in (
    ("morphological_open", "morphological_open_set"),
    ("decomposition", "decomposition_set"),
):
    for familiarity in ("training_seen", "training_unseen"):
        tp = fp = fn = 0

        for gold_set, predicted_set in zip(
            linking_audit["gold_set"],
            linking_audit[prediction_column],
        ):
            if familiarity == "training_seen":
                reference = gold_set & seen_labels
                prediction = predicted_set & seen_labels
            else:
                reference = gold_set - seen_labels
                prediction = predicted_set - seen_labels

            tp += len(reference & prediction)
            fp += len(prediction - reference)
            fn += len(reference - prediction)

        rows.append({
            "gold_familiarity": familiarity,
            "system": system,
            "gold_links": tp + fn,
            "predicted_links": tp + fp,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "micro_F1": 2 * tp / (2 * tp + fp + fn),
        })

familiarity_scores = pd.DataFrame(rows)
display(familiarity_scores.round({"micro_F1": 4}))

for system in familiarity_scores["system"].unique():
    subset = familiarity_scores[familiarity_scores["system"] == system]
    assert subset["gold_links"].sum() == 131
    assert set(subset["gold_links"]) == {100, 31}